### Qwen 2.5 5B Streaming and Batching with vLLM

In [ ]:
# Upgrade necessary packages
import importlib.util
import torch

if importlib.util.find_spec("google.colab"):
    %pip uninstall -y torchaudio
    # install packages into the current Jupyter kernel's Python environment.
    %pip install -q vllm transformers tiktoken

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("vLLM imported successfully")

In [ ]:
import torch
import gc
import time

# Unload models and clean up gpu memory cache
def free_gpu(model):
  if model:
    # Removes the reference to the model's memory,
    # making it eligible for garbage collection.
    del model

  # Release any cached GPU memory that's no longer needed.
  if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

  # Trigger garbage collection to ensure memory is fully released.
  gc.collect()

free_gpu(None)

#### Batching multiple prompts
Helps with throughput and overall latency.

In [ ]:
# Prompts for batch generation, 4 input sequences
import time
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen2.5-0.5B"

# Load model with vLLM.
llm = LLM(
  model=model_name, 
  dtype="float16",
  gpu_memory_utilization=0.7)  # Prevents OOM on Colab GPUs

prompts = [
    "Once upon a time, in a land far away, there was a",
    "In the future, humanity will",
    "The quick brown fox jumps over the lazy",
    "Artificial intelligence is transforming the world by"
]

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.9,  # Nucleus sampling parameter
    max_tokens=100
)

# Process 4 input sequences together in one batch
start_time = time.time()

vllm_outputs = llm.generate(prompts=prompts, sampling_params=sampling_params)

end_time = time.time()

vllm_time = end_time - start_time
print(f"Batch generation took {vllm_time:.4f} seconds")

# Process prompts one by one
start_time = time.time()

for prompt in prompts:
    vllm_outputs = llm.generate(prompts=[prompt], sampling_params=sampling_params)

end_time = time.time()

vllm_time = end_time - start_time
print(f"Single generation took {vllm_time:.4f} seconds")


#### Define Prompt

In [ ]:

# Define the prompt.
prompt = """You are an expert AI historian writing a detailed chapter for a book titled "The Evolution of Human-AI Collaboration."
Begin by summarizing the early stages of artificial intelligence in the 1950s, touching on symbolic logic and rule-based systems. Then transition into the rise of machine learning, particularly deep learning in the 2010s.
Afterward, describe how large language models like GPT transformed human-computer interaction, enabling applications in education, creative writing, customer support, and software development.
Finally, reflect on the societal and ethical implications of AI, such as misinformation, bias, and the alignment problem.
Write in a formal tone, with rich detail and examples in each era."""

#### Streaming
1. Return result when generation completes.
2. Return as soon as we have a token.

In [ ]:
import asyncio
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
from vllm.sampling_params import SamplingParams

# Initialize the engine arguments
engine_args = AsyncEngineArgs(
    model="Qwen/Qwen2.5-0.5B",
    dtype="float16",
    tensor_parallel_size=1,      # Number of GPUs to use
    gpu_memory_utilization=0.7,  # GPU memory utilization
    max_num_batched_tokens=32768, # Maximum number of tokens to process in a batch
    max_num_seqs=256,           # Maximum number of sequences to process
    enable_log_requests=True,   # Enable request logging
    disable_log_stats=True,      # Disable stats logging
)

# Create the vLLM async streaming engine
engine = AsyncLLMEngine.from_engine_args(engine_args)


async def generate_text(prompt: str, max_tokens: int = 100, request_id="id"):
  try:
    # Define sampling parameters
    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=max_tokens,
        stop=["\n"],  # Stop at newline
    )

    # Generate text in async and streaming fashion
    results_generator = engine.generate(
        prompt=prompt,
        sampling_params=sampling_params,
        request_id=request_id
    )

    # Process the results
    final_output = None
    async for request_output in results_generator:
        final_output = request_output
        # Print each token as it's generated
        print("chunk \n")
        for output in request_output.outputs:
            print(output.text, end="", flush=True)
        print()
    print()  # Newline at the end

    # This will only be reached if all tokens are generated
    print("\nGeneration completed successfully")

    return final_output
  except asyncio.CancelledError:
    print("\nGeneration was cancelled")
    return None
  finally:
    # Always clean up
    try:
      await engine.abort(request_id)
    except:
      pass


request_id = "any_id"
await generate_text(prompt=prompt, max_tokens=10000, request_id=request_id)